In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

budget_data = [
    ("Health", 2019, 750.0),
    ("Education", 2019, 500.0),
    ("Health", 2020, 800.0),
    ("Education", 2020, 550.0),
]

spending_data = [
    ("Health", 2019, 700.0),
    ("Education", 2019, 450.0),
    ("Health", 2020, 780.0),
    ("Education", 2020, 540.0),
]

budget_df = spark.createDataFrame(budget_data, schema=["Department", "Year", "Budget"])
spending_df = spark.createDataFrame(
    spending_data, schema=["Department", "Year", "Spending"]
)

budget_df.show()
spending_df.show()

+----------+----+------+
|Department|Year|Budget|
+----------+----+------+
|    Health|2019| 750.0|
| Education|2019| 500.0|
|    Health|2020| 800.0|
| Education|2020| 550.0|
+----------+----+------+

+----------+----+--------+
|Department|Year|Spending|
+----------+----+--------+
|    Health|2019|   700.0|
| Education|2019|   450.0|
|    Health|2020|   780.0|
| Education|2020|   540.0|
+----------+----+--------+



In [5]:
window = Window.partitionBy("Department")

# Calculate variance for Budget and Spending in each department over the years
budget_variance = budget_df.withColumn(
    "Budget_Variance",
    variance("Budget").over(window),
)

spending_variance = spending_df.withColumn(
    "Spending_Variance",
    variance("Spending").over(window).cast("integer"),
)

# Deduplicate rows and join the variance DataFrames
budget_variance = budget_variance.dropDuplicates(["Department"])
spending_variance = spending_variance.dropDuplicates(["Department"])

budget_variance.join(
    spending_variance,
    on=["Department"],
    how="inner",
).select(
    "Department",
    "Budget_Variance",
    "Spending_Variance",
).show()

+----------+---------------+-----------------+
|Department|Budget_Variance|Spending_Variance|
+----------+---------------+-----------------+
| Education|         1250.0|             4050|
|    Health|         1250.0|             3200|
+----------+---------------+-----------------+

